# Fine-tune Qwen2.5-0.5B-Instruct for Aaron's Portfolio Chat

QLoRA fine-tune → ONNX export → quantize → upload to HuggingFace.

**Runtime:** GPU → T4 (free tier works). Training takes ~5 minutes.

In [ ]:
!pip install -q torch transformers datasets peft accelerate "optimum[onnxruntime]" bitsandbytes sentencepiece huggingface_hub onnx onnxruntime

In [ ]:
import os, shutil

# Locate dataset_v2.jsonl — check CWD, then /kaggle/input/*/
if not os.path.exists('dataset_v2.jsonl'):
    found = None
    kaggle_root = '/kaggle/input'
    if os.path.isdir(kaggle_root):
        for root, _, files_in_dir in os.walk(kaggle_root):
            if 'dataset_v2.jsonl' in files_in_dir:
                found = os.path.join(root, 'dataset_v2.jsonl')
                break
    if not found:
        raise FileNotFoundError(
            "dataset_v2.jsonl not found. Attach it via '+ Add Input' in the Kaggle sidebar, "
            "or place it in the working directory."
        )
    shutil.copy(found, 'dataset_v2.jsonl')
    print(f"Copied from {found}")

with open('dataset_v2.jsonl') as f:
    lines = [l for l in f if l.strip()]
print(f"Dataset: {len(lines)} examples")

In [ ]:
import json, sys, time
import torch
from datasets import Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

# ─── HARD GPU ASSERTION — fail loud if no CUDA, don't silently fall to CPU ───
assert torch.cuda.is_available(), (
    "NO GPU DETECTED. Kaggle: right sidebar → Session options → "
    "Accelerator → GPU T4 x2, then restart the session."
)
print(f">>> GPU: {torch.cuda.get_device_name(0)}", flush=True)
print(f">>> CUDA: {torch.version.cuda}  torch: {torch.__version__}", flush=True)

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "./output"
MERGED_DIR = "./merged"
ONNX_DIR = "./onnx_export"
QUANTIZED_DIR = "./onnx_quantized"
MAX_SEQ_LENGTH = 768

# Must match src/constants/aaronChatFacts.js AARON_CHAT_SYSTEM_PROMPT exactly.
SYSTEM_PROMPT_BASE = (
    "You are a helpful assistant on Aaron Rohrbacher's portfolio site. "
    "Answer the question using ONLY the provided facts. One to two sentences. "
    "Do not add information not in the facts. "
    "If the facts don't cover the question, say: "
    "\"I don't have that info — just say 'connect me' and I'll open a live chat with Aaron!\""
)

# Must match src/constants/aaronChatFacts.js FACT_CHUNKS exactly.
FACT_CHUNKS = [
    "Aaron Rohrbacher is a Lead AI/ML Software Engineer and DevOps Architect based in Portland, Oregon. He wrapped up his role as Lead Software Development Engineer at Forbes AAC in March 2026 and is actively seeking his next role, available to start immediately.",
    "At Forbes AAC, Aaron led stabilization of a legacy Ruby and Ember stack and an enterprise rebuild using Next.js. He rebuilt native apps: iOS in Swift, Android in Java and Kotlin, macOS in Swift, Windows and Linux in Qt and Rust.",
    "At SPARQ, Aaron was technical lead on enterprise AI projects, AWS cloud migrations, and conversational AI. He worked on payroll and logistics systems at scale serving over 500,000 employees.",
    "Aaron's earlier employers include Nuel Cloud (AWS LAMP provisioning), Nordic, Fiduciary Benchmarks, and Planet Argon.",
    "Aaron's programming languages include JavaScript, TypeScript, Python, Ruby, Java, Kotlin, Swift, Rust, PHP, SQL, and Bash.",
    "Aaron works across AWS, GCP, and Azure. He holds AWS Cloud Practitioner and Developer Associate certifications and is pursuing AWS DevOps Professional.",
    "Aaron's AI and ML stack includes PyTorch, LLM fine-tuning, NLP, and Amazon AI services: Lex, Polly, Transcribe, and Q.",
    "Aaron's infrastructure and DevOps tools include Docker, Kubernetes, Terraform, AWS CDK, and SST.",
    "Aaron built this portfolio site using Next.js, SST, and AWS. Other projects: Klear (a KWin window manager plugin), Thinger (Python video processing), a PyTorch GPT-2 trained on George Carlin transcripts, and the Nuel API.",
    "Aaron plays saxophone and is a musician outside of his engineering career.",
]
FACTS_BLOCK = "Facts about Aaron:\n" + "\n".join(f"- {c}" for c in FACT_CHUNKS)
SYSTEM_PROMPT = SYSTEM_PROMPT_BASE + "\n\n" + FACTS_BLOCK

# --- Load dataset ---
records = []
with open('dataset_v2.jsonl') as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))
raw_dataset = Dataset.from_list(records)
print(f">>> Loaded {len(raw_dataset)} examples", flush=True)

# --- Tokenizer ---
# fix_mistral_regex=True repairs a known tokenizer regex issue transformers
# warns about even on Qwen-family tokenizers. Without it, the tokenizer saved
# into MERGED_DIR emits warnings (and potential mis-tokenization) at export.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, fix_mistral_regex=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(">>> Tokenizer loaded", flush=True)

# --- Model + LoRA ---
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.bfloat16, device_map="auto")
model.config.use_cache = False
print(f">>> Model loaded on: {next(model.parameters()).device}", flush=True)
assert next(model.parameters()).is_cuda, "Model not on CUDA after load — device_map='auto' failed."

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=32, lora_alpha=64, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f">>> PEFT model device: {next(model.parameters()).device}", flush=True)

# --- Format + tokenize ---
def format_example(example):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + example["messages"]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

formatted = [format_example(ex) for ex in raw_dataset]
ds = Dataset.from_dict({"text": formatted})
ds = concatenate_datasets([ds, ds.shuffle(seed=42), ds.shuffle(seed=99), ds.shuffle(seed=7)])  # 4x

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_SEQ_LENGTH, padding=False)

tokenized = ds.map(tokenize_fn, batched=True, remove_columns=["text"])
print(f">>> Training on {len(tokenized)} tokenized examples", flush=True)

# --- Loud callback: prints every step + every epoch to stdout with flush ---
class LoudCallback(TrainerCallback):
    def __init__(self):
        self.start = None
        self.epoch_start = None
    def on_train_begin(self, args, state, control, **kw):
        self.start = time.time()
        print(f">>> TRAIN START — total_steps={state.max_steps} "
              f"epochs={args.num_train_epochs} "
              f"eff_batch={args.per_device_train_batch_size * args.gradient_accumulation_steps}",
              flush=True)
        sys.stdout.flush()
    def on_epoch_begin(self, args, state, control, **kw):
        self.epoch_start = time.time()
        epoch_num = int(state.epoch) + 1 if state.epoch is not None else 1
        print(f">>> ===== EPOCH {epoch_num}/{int(args.num_train_epochs)} BEGIN "
              f"(step {state.global_step}/{state.max_steps}) =====", flush=True)
        sys.stdout.flush()
    def on_epoch_end(self, args, state, control, **kw):
        epoch_num = int(round(state.epoch)) if state.epoch is not None else 0
        epoch_time = time.time() - self.epoch_start if self.epoch_start else 0
        total_elapsed = time.time() - self.start
        print(f">>> ===== EPOCH {epoch_num}/{int(args.num_train_epochs)} END "
              f"epoch_time={epoch_time:.0f}s  total_elapsed={total_elapsed:.0f}s "
              f"(step {state.global_step}/{state.max_steps}) =====", flush=True)
        sys.stdout.flush()
    def on_step_end(self, args, state, control, **kw):
        if state.global_step % 5 == 0 or state.global_step <= 3:
            elapsed = time.time() - self.start
            rate = state.global_step / elapsed if elapsed > 0 else 0
            eta = (state.max_steps - state.global_step) / rate if rate > 0 else 0
            print(f">>> step {state.global_step}/{state.max_steps} "
                  f"epoch={state.epoch:.2f} elapsed={elapsed:.0f}s eta={eta:.0f}s", flush=True)
            sys.stdout.flush()
    def on_log(self, args, state, control, logs=None, **kw):
        if logs and "loss" in logs:
            print(f">>> LOG step={state.global_step} epoch={state.epoch:.2f} "
                  f"loss={logs['loss']:.4f} lr={logs.get('learning_rate', 0):.2e}", flush=True)
            sys.stdout.flush()

# --- Train ---
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=10,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        logging_steps=5,
        logging_first_step=True,
        save_strategy="epoch",
        save_total_limit=1,
        report_to="none",
        disable_tqdm=True,            # tqdm's \r updates get buffered on Kaggle
        dataloader_num_workers=0,     # Kaggle deadlocks with >0 workers
        dataloader_pin_memory=True,
    ),
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=[LoudCallback()],
)

print(">>> Starting trainer.train() ...", flush=True)
sys.stdout.flush()
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(">>> Done training.", flush=True)

In [ ]:
# Merge LoRA into base model
print("Merging LoRA weights...")
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.bfloat16, device_map="cpu")
merged = PeftModel.from_pretrained(base, OUTPUT_DIR)
merged = merged.merge_and_unload()
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model: {MERGED_DIR}/")

In [ ]:
# Sanity check — test a few questions before exporting
from transformers import pipeline as hf_pipeline

pipe = hf_pipeline("text-generation", model=MERGED_DIR, tokenizer=MERGED_DIR, dtype=torch.bfloat16, device_map="auto")

for q in ["Who is Aaron?", "What languages does he know?", "What is the meaning of life?", "Does he know Rust?", "How do I contact Aaron?"]:
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": q}]
    out = pipe(msgs, max_new_tokens=60, do_sample=False, repetition_penalty=1.3)
    print(f"Q: {q}")
    print(f"A: {out[0]['generated_text'][-1]['content']}\n")

In [ ]:
# Export to ONNX
from optimum.exporters.onnx import main_export

# Repair the MERGED_DIR tokenizer with fix_mistral_regex=True, then re-save.
# If training ran before this flag was added to cell-3, the tokenizer currently
# on disk has the broken regex and optimum will warn repeatedly during export.
print("Repairing tokenizer regex in MERGED_DIR...")
_tok_fix = AutoTokenizer.from_pretrained(MERGED_DIR, fix_mistral_regex=True)
_tok_fix.save_pretrained(MERGED_DIR)
del _tok_fix

print("Exporting to ONNX...")
main_export(
    model_name_or_path=MERGED_DIR,
    output=ONNX_DIR,
    task="text-generation-with-past",
    device="cuda",
    fp16=False,
)
print(f"ONNX export: {ONNX_DIR}/")

In [ ]:
# Quantize (INT8 dynamic) to reduce model size
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig
import glob as globmod

print("Quantizing...")
os.makedirs(QUANTIZED_DIR, exist_ok=True)

# Quantize each ONNX file
onnx_files = globmod.glob(os.path.join(ONNX_DIR, "*.onnx"))
for onnx_path in onnx_files:
    quantizer = ORTQuantizer.from_pretrained(ONNX_DIR, file_name=os.path.basename(onnx_path))
    qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)
    quantizer.quantize(save_dir=QUANTIZED_DIR, quantization_config=qconfig)

# Copy config + tokenizer files. Exclude .onnx AND .onnx_data: the latter is
# external weights for the unquantized model and would be an orphaned ~2GB file
# in the output dir since we don't copy model.onnx itself.
for f in os.listdir(ONNX_DIR):
    src = os.path.join(ONNX_DIR, f)
    dst = os.path.join(QUANTIZED_DIR, f)
    if not os.path.isfile(src):
        continue
    if f.endswith('.onnx') or f.endswith('.onnx_data'):
        continue
    if os.path.exists(dst):
        continue
    shutil.copy2(src, dst)

import subprocess
size = subprocess.run(['du', '-sh', QUANTIZED_DIR], capture_output=True, text=True).stdout.strip()
print(f"Quantized model: {size}")

In [ ]:
# Zip the quantized model for download
import zipfile

# On Kaggle, write to /kaggle/working/ so it appears in the notebook's Output tab.
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
zip_path = os.path.join(OUT_DIR, "onnx_quantized.zip")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files_list in os.walk(QUANTIZED_DIR):
        for fname in files_list:
            full = os.path.join(root, fname)
            arcname = os.path.relpath(full, QUANTIZED_DIR)
            zf.write(full, arcname)

print(f"Zipped to {zip_path}")

# Colab: trigger browser download. Kaggle: skip (file is in Output tab).
try:
    from google.colab import files as _colab_files
    _colab_files.download(zip_path)
except ImportError:
    print("On Kaggle — download from the notebook's Output tab (right sidebar) after session ends.")